<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 2 · Python Data Analysis, Visualization, and Storytelling
### Notebook 04 · Basic SQL Awareness

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
The point of this notebook is to keep relational thinking visible: select,
filter, group, and join. The SQL helper is loaded directly from the book
local `code/` folder so the path mechanics stay explicit.


This cell continues the worked example for the current section.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "2_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util

module_path = BOOK_ROOT / "code" / "sql_demo.py"
spec = importlib.util.spec_from_file_location("sql_demo", module_path)
sql_demo = importlib.util.module_from_spec(spec)
assert spec and spec.loader is not None
spec.loader.exec_module(sql_demo)

build_demo = sql_demo.build_demo

conn, cur, orders, customers = build_demo()

print(orders)
print()
print(customers)


This cell continues the worked example for the current section.


In [ ]:
sql_select = '''
SELECT
    order_id,
    amount
FROM orders;
'''

sql_filter = '''
SELECT
    order_id,
    amount
FROM orders
WHERE amount >= 100.0;
'''

print(cur.execute(sql_select).fetchall())
print()
print(cur.execute(sql_filter).fetchall())


This cell groups the data and orders the result so the learner can compare categories clearly.


In [ ]:
sql_group = '''
SELECT
    c.country,
    SUM(o.amount) AS total_amount
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY
    c.country
ORDER BY
    total_amount DESC;
'''

sql_result = cur.execute(sql_group).fetchall()
print(sql_result)

pandas_result = (
    orders.merge(customers, on='customer_id', how='left')
    .groupby('country')['amount']
    .sum()
    .sort_values(ascending=False)
)
print()
print(pandas_result)


This cell continues the worked example for the current section.


In [ ]:
artifact_dir = BOOK_ROOT / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

note_path = artifact_dir / 'sql_awareness_note.txt'
note_path.write_text(
    'SQL and pandas express the same table logic in different syntax.\n'
    'SELECT matches column selection, WHERE matches filtering, GROUP BY\n'
    'matches aggregation, and JOIN matches merge.\n'
)

conn.close()
print(note_path.resolve())


<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
